In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

sales = pd.read_excel("../data/Online Retail.xlsx")

sales.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Data Cleaning

Before starting the analysis, the dataset was inspected for unusual values, missing values, and records that could affect the accuracy of the sales analysis.

The cleaning decisions were made based on the meaning of the data rather than automatically removing unusual or missing values.

### 1. Negative Quantities

Negative quantities were investigated because they can represent returns or cancelled transactions rather than incorrect data.

In [2]:
(sales["Quantity"] < 0).sum()

np.int64(10624)

Most negative quantities were associated with cancellation invoices. These were identified by invoice numbers beginning with `C`.

Cancellation records were kept because they represent meaningful transaction information.

In [3]:
sales["InvoiceNo"].str.startswith("C").sum()

9288

The remaining negative-quantity records were investigated to determine whether they represented meaningful sales activity.

In [4]:
sales[
    (sales["Quantity"] < 0) &
    (~sales["InvoiceNo"].str.startswith("C", na=False))
].shape

(1336, 8)

In [5]:
sales_clean = sales[
    ~(
        (sales["Quantity"] < 0) &
        (sales["UnitPrice"] == 0)
    )
].copy()

### 2. Negative Unit Prices

Negative unit prices were investigated because a product sale should not normally have a negative selling price.

In [6]:
(sales_clean["UnitPrice"] < 0).sum()

np.int64(2)

Both records were labelled `Adjust bad debt`, indicating accounting adjustments rather than actual product sales.

Therefore, these records were removed from the sales analysis.

In [7]:
sales_clean = sales_clean[
    sales_clean["UnitPrice"] >= 0
].copy()

### 3. Missing CustomerID

Missing `CustomerID` values were investigated rather than automatically removed.

These records can still be useful for overall sales, product, country, and time-based analysis. Therefore, they were retained.

For customer-level analysis, records with a known `CustomerID` will be used.

In [8]:
sales_clean["CustomerID"].isna().sum()

np.int64(133742)

### 4. Missing Product Descriptions

Missing product descriptions were investigated to determine whether these records still contained useful sales information.

In [9]:
sales_clean["Description"].isna().sum()

np.int64(592)

In [10]:
sales_clean[
    sales_clean["Description"].isna() &
    sales_clean["CustomerID"].isna() &
    (sales_clean["UnitPrice"] == 0)
].shape[0]

592

All 592 records with missing `Description` also had a missing `CustomerID` and a `UnitPrice` of 0.

Since these records provide no identifiable product, customer, or revenue information for our sales analysis, they were removed.

In [11]:
sales_clean = sales_clean[
    ~(
        sales_clean["Description"].isna() &
        sales_clean["CustomerID"].isna() &
        (sales_clean["UnitPrice"] == 0)
    )
].copy()

### 5. Duplicate Records

Exact duplicate rows were checked across all columns. We found **5,268 duplicate records**.

After investigating the duplicate rows, we confirmed that they were exact duplicates, meaning all column values were identical. Therefore, these duplicate rows were **removed** from the dataset to avoid counting the same transaction record multiple times.

```python
sales_clean = sales_clean.drop_duplicates().copy()
```

The dataset was then verified to ensure that no exact duplicate rows remained.


In [12]:
sales_clean=sales_clean.drop_duplicates().copy()

In [13]:
sales_clean.duplicated().sum()

np.int64(0)

In [14]:
sales_clean["total_sales"]=sales_clean["Quantity"] * sales_clean["UnitPrice"]

In [15]:
sales_clean[["total_sales","Quantity","UnitPrice"]].head()

,total_sales,Quantity,UnitPrice
0,15.30,6,2.55
1,20.34,6,3.39
2,22.00,8,2.75
3,20.34,6,3.39
4,20.34,6,3.39


In [16]:
sales_clean["Year"] = sales_clean["InvoiceDate"].dt.year
sales_clean["Month"] = sales_clean["InvoiceDate"].dt.month
sales_clean["DayOfWeek"] = sales_clean["InvoiceDate"].dt.day_name()
sales_clean["Hour"] = sales_clean["InvoiceDate"].dt.hour

In [17]:
sales_clean[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour"]].head()

,InvoiceDate,Year,Month,DayOfWeek,Hour
0,2010-12-01 08:26:00,2010,12,Wednesday,8
1,2010-12-01 08:26:00,2010,12,Wednesday,8
2,2010-12-01 08:26:00,2010,12,Wednesday,8
3,2010-12-01 08:26:00,2010,12,Wednesday,8
4,2010-12-01 08:26:00,2010,12,Wednesday,8


In [18]:
sales_clean["MonthYear"] = sales_clean["InvoiceDate"].dt.to_period("M")

In [19]:
sales_clean[["InvoiceDate", "MonthYear"]].head()

,InvoiceDate,MonthYear
0,2010-12-01 08:26:00,2010-12
1,2010-12-01 08:26:00,2010-12
2,2010-12-01 08:26:00,2010-12
3,2010-12-01 08:26:00,2010-12
4,2010-12-01 08:26:00,2010-12


In [20]:
total_sales = sales_clean["total_sales"].sum()
total_sales

np.float64(9748131.074000001)

In [21]:
yearly_sales = sales_clean.groupby("Year")["total_sales"].sum()
yearly_sales

Year
2010     746723.610
2011    9001407.464
Name: total_sales, dtype: float64

In [22]:
country_sales = sales_clean.groupby("Country")["total_sales"].sum().sort_values(ascending=False)
country_sales

Country
United Kingdom          8189252.304
Netherlands              284661.540
EIRE                     262993.380
Germany                  221509.470
France                   197317.110
Australia                137009.770
Switzerland               56363.050
Spain                     54756.030
Belgium                   40910.960
Sweden                    36585.410
Japan                     35340.620
Norway                    35163.460
Portugal                  29302.970
Finland                   22326.740
Channel Islands           20076.390
Denmark                   18768.140
Italy                     16890.510
Cyprus                    12858.760
Austria                   10154.320
Hong Kong                  9908.240
Singapore                  9120.390
Israel                     7901.970
Poland                     7213.140
Unspecified                4740.940
Greece                     4710.520
Iceland                    4310.000
Canada                     3666.380
Malta               

In [23]:
product_sales = (
    sales_clean.groupby("Description")["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

product_sales

Description
DOTCOM POSTAGE                        206245.480
REGENCY CAKESTAND 3 TIER              164459.490
WHITE HANGING HEART T-LIGHT HOLDER     99612.420
PARTY BUNTING                          98243.880
JUMBO BAG RED RETROSPOT                92175.790
                                         ...    
Discount                               -5696.220
Bank Charges                           -7175.639
CRUK Commission                        -7933.430
Manual                                -69031.640
AMAZON FEE                           -221520.500
Name: total_sales, Length: 4093, dtype: float64

In [24]:
monthly_sales = (
    sales_clean.groupby("MonthYear")["total_sales"]
    .sum()
)

monthly_sales

MonthYear
2010-12     746723.610
2011-01     558448.560
2011-02     497026.410
2011-03     682013.980
2011-04     492367.841
2011-05     722094.100
2011-06     689977.230
2011-07     680156.991
2011-08     703510.580
2011-09    1017596.682
2011-10    1069368.230
2011-11    1456145.800
2011-12     432701.060
Freq: M, Name: total_sales, dtype: float64

In [25]:
sales_clean.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'total_sales', 'Year', 'Month',
       'DayOfWeek', 'Hour', 'MonthYear'],
      dtype='str')

In [26]:
daily_sales = (
    sales_clean.groupby("DayOfWeek")["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

daily_sales

DayOfWeek
Thursday     2108701.530
Tuesday      1965703.611
Wednesday    1730088.430
Monday       1584895.301
Friday       1560082.741
Sunday        798659.461
Name: total_sales, dtype: float64

In [27]:
sales_clean.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'total_sales', 'Year', 'Month',
       'DayOfWeek', 'Hour', 'MonthYear'],
      dtype='str')

In [28]:
hourly_sales = (
    sales_clean.groupby("Hour")["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

hourly_sales

Hour
12    1357595.120
10    1327329.891
15    1186819.410
13    1172985.870
11    1146457.490
14    1113532.861
9      766524.171
16     727666.670
17     434834.541
8      281723.020
18     140365.040
19      45864.930
7       31009.320
20      15920.090
6        -497.350
Name: total_sales, dtype: float64

In [29]:

cancelled_sales = sales_clean.loc[
    sales_clean["InvoiceNo"].str.startswith("C" ,na=False),
    "total_sales"
].sum()
net_sales = total_sales + cancelled_sales
net_sales

np.float64(8854151.344)

In [30]:
customer_sales = (
    sales_clean.groupby("CustomerID")["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

customer_sales

CustomerID
14646.0    279489.02
18102.0    256438.49
17450.0    187322.17
14911.0    132458.73
12415.0    123725.45
             ...    
12503.0     -1126.00
17603.0     -1165.30
14213.0     -1192.20
15369.0     -1592.49
17448.0     -4287.63
Name: total_sales, Length: 4372, dtype: float64

In [31]:
invoice_sales = (
    sales_clean.groupby("InvoiceNo")["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

invoice_sales

InvoiceNo
581483     168469.60
541431      77183.60
574941      52940.94
576365      50653.91
556444      38970.00
             ...    
C580605    -17836.46
C550456    -22998.40
C556445    -38970.00
C541433    -77183.60
C581484   -168469.60
Name: total_sales, Length: 23970, dtype: float64

In [32]:
highest_sold_product=sales_clean.groupby("Description")["Quantity"].sum().sort_values(ascending=False)
highest_sold_product

Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS    53751
JUMBO BAG RED RETROSPOT              47260
POPCORN HOLDER                       36322
ASSORTED COLOUR BIRD ORNAMENT        36282
PACK OF 72 RETROSPOT CAKE CASES      36016
                                     ...  
CRUK Commission                        -16
ASSORTED TUTTI FRUTTI ROUND BOX        -24
AMAZON FEE                             -30
SAMPLES                                -58
Discount                             -1194
Name: Quantity, Length: 4093, dtype: int64

In [33]:
average_order_value = invoice_sales.mean()

average_order_value

np.float64(406.68047868168543)

In [34]:
invoice_quantity = (
    sales_clean.groupby("InvoiceNo")["Quantity"]
    .sum()
)

average_quantity_per_order = invoice_quantity.mean()

average_quantity_per_order

np.float64(222.64956195244056)

In [35]:
customer_orders = (
    sales_clean.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .sort_values()
)

customer_orders

CustomerID
18278.0      1
18262.0      1
12361.0      1
12357.0      1
12355.0      1
          ... 
13089.0    118
14606.0    128
17841.0    169
12748.0    224
14911.0    248
Name: InvoiceNo, Length: 4372, dtype: int64

In [36]:
one_time_customers = customer_orders[customer_orders == 1]
repeat_customers = customer_orders[customer_orders > 1]
one_time_count = one_time_customers.count()
repeat_count = repeat_customers.count()

one_time_count, repeat_count

(np.int64(1313), np.int64(3059))

In [43]:
top_10_customer_sales=customer_sales.head(10)
top_10_customer_sales_percentage=top_10_customer_sales.sum()
top_10_customer_sales_percentage=top_10_customer_sales_percentage/total_sales *100
top_10_customer_sales_percentage
other_customer_percentage=100 - top_10_customer_sales_percentage
other_customer_percentage,top_10_customer_sales_percentage



(np.float64(85.95996884315181), np.float64(14.04003115684819))

In [44]:
monthly_orders = (
    sales_clean.groupby("MonthYear")["InvoiceNo"]
    .nunique()
)

monthly_orders

MonthYear
2010-12    1892
2011-01    1349
2011-02    1320
2011-03    1777
2011-04    1495
2011-05    2003
2011-06    1867
2011-07    1755
2011-08    1643
2011-09    2191
2011-10    2432
2011-11    3271
2011-12     975
Freq: M, Name: InvoiceNo, dtype: int64

In [45]:
monthly_average_order_value = monthly_sales / monthly_orders

monthly_average_order_value

MonthYear
2010-12    394.674212
2011-01    413.972246
2011-02    376.535159
2011-03    383.800777
2011-04    329.343037
2011-05    360.506291
2011-06    369.564665
2011-07    387.553841
2011-08    428.186598
2011-09    464.443944
2011-10    439.707331
2011-11    445.168389
2011-12    443.795959
Freq: M, dtype: float64

In [51]:
average_orders_per_customer = customer_orders.mean()

average_orders_per_customer

np.float64(5.07548032936871)

In [54]:
cancelled_products = (
    sales_clean.loc[
        sales_clean["InvoiceNo"].str.startswith("C", na=False)
    ]
    .groupby("Description")["Quantity"]
    .sum()
    .sort_values()
)

cancelled_products.head(10)

Description
PAPER CRAFT , LITTLE BIRDIE           -80995
MEDIUM CERAMIC TOP STORAGE JAR        -74494
ROTATING SILVER ANGELS T-LIGHT HLDR    -9376
Manual                                 -4066
FAIRY CAKE FLANNEL ASSORTED COLOUR     -3150
WHITE HANGING HEART T-LIGHT HOLDER     -2578
GIN + TONIC DIET METAL SIGN            -2030
HERB MARKER BASIL                      -1527
FELTCRAFT DOLL MOLLY                   -1447
TEA TIME PARTY BUNTING                 -1424
Name: Quantity, dtype: int64

In [67]:
country_orders = (
    sales_clean.groupby("Country")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
)

country_orders

Country
United Kingdom          21564
Germany                   603
France                    461
EIRE                      360
Belgium                   119
Spain                     105
Netherlands               101
Switzerland                74
Portugal                   71
Australia                  69
Italy                      55
Finland                    48
Sweden                     46
Norway                     40
Channel Islands            33
Japan                      28
Poland                     24
Denmark                    21
Cyprus                     20
Austria                    19
Hong Kong                  15
Unspecified                13
Malta                      10
Singapore                  10
Israel                      9
Iceland                     7
USA                         7
Greece                      6
Canada                      6
Czech Republic              5
European Community          5
Bahrain                     4
Lithuania                   4
Un